Healthcare Emergency Department Performance Analysis

Purpose:
Inspect and prepare three CMS datasets for an analysis of emergency
department performance in Mississippi, Louisiana, and Alabama.

Datasets:
1. Hospital General Information
2. Timely and Effective Care — Hospital
3. Timely and Effective Care — State

This notebook will document data inspection, cleaning, validation,
and the export of prepared datasets for MySQL and Tableau.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Hospital_General_Information.csv to Hospital_General_Information.csv
Saving Timely_and_Effective_Care-Hospital.csv to Timely_and_Effective_Care-Hospital.csv
Saving Timely_and_Effective_Care-State.csv to Timely_and_Effective_Care-State.csv


In [ ]:
import pandas as pd
import io

raw_data = {}

for filename, contents in uploaded.items():
    if filename.lower().endswith(".csv"):
        raw_data[filename] = pd.read_csv(
            io.BytesIO(contents),
            dtype="string",
            keep_default_na=False
        )

print(f"CSV files loaded: {len(raw_data)}")

CSV files loaded: 3


In [ ]:
for filename, df in raw_data.items():
    print(f"\nFILE: {filename}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    print("\nCOLUMN NAMES:")
    for column in df.columns:
        print(column)

    print("\nFIRST THREE ROWS:")
    display(df.head(3))


FILE: Hospital_General_Information.csv
Rows: 5,419
Columns: 38

COLUMN NAMES:
Facility ID
Facility Name
Address
City/Town
State
ZIP Code
County/Parish
Telephone Number
Hospital Type
Hospital Ownership
Emergency Services
Meets criteria for birthing friendly designation
Hospital overall rating
Hospital overall rating footnote
MORT Group Measure Count
Count of Facility MORT Measures
Count of MORT Measures Better
Count of MORT Measures No Different
Count of MORT Measures Worse
MORT Group Footnote
Safety Group Measure Count
Count of Facility Safety Measures
Count of Safety Measures Better
Count of Safety Measures No Different
Count of Safety Measures Worse
Safety Group Footnote
READM Group Measure Count
Count of Facility READM Measures
Count of READM Measures Better
Count of READM Measures No Different
Count of READM Measures Worse
READM Group Footnote
Pt Exp Group Measure Count
Count of Facility Pt Exp Measures
Pt Exp Group Footnote
TE Group Measure Count
Count of Facility TE Measures
TE 

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,,15,15,,10,10,
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,,15,15,,10,10,
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,,15,15,,10,9,



FILE: Timely_and_Effective_Care-Hospital.csv
Rows: 138,084
Columns: 16

COLUMN NAMES:
Facility ID
Facility Name
Address
City/Town
State
ZIP Code
County/Parish
Telephone Number
Condition
Measure ID
Measure Name
Score
Sample
Footnote
Start Date
End Date

FIRST THREE ROWS:


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Condition,Measure ID,Measure Name,Score,Sample,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Emergency Department,EDV,Emergency department volume,very high,,,01/01/2024,12/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS,Global Malnutrition Composite Score,Not Available,Not Available,5,01/01/2024,12/31/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024



FILE: Timely_and_Effective_Care-State.csv
Rows: 1,736
Columns: 8

COLUMN NAMES:
State
Condition
Measure ID
Measure Name
Score
Footnote
Start Date
End Date

FIRST THREE ROWS:


,State,Condition,Measure ID,Measure Name,Score,Footnote,Start Date,End Date
0,AK,Healthcare Personnel Vaccination,IMM_3,Healthcare workers given influenza vaccination...,74,,10/01/2024,03/31/2025
1,AK,Emergency Department,OP_18a,Average (median) time all patients spent in th...,143,,10/01/2024,09/30/2025
2,AK,Emergency Department,OP_18a_HIGH_MIN,Average (median) time all patients spent in th...,156,,10/01/2024,09/30/2025


Column Standardization and Measure Availability

Create working copies of the source tables with consistent column names.
Check the availability and reporting periods of OP_18b and OP_22 for
Mississippi, Louisiana, and Alabama before preparing comparisons.

In [ ]:
def standardize_columns(df):
    working_copy = df.copy()

    working_copy.columns = (
        working_copy.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    return working_copy


hospital_general = standardize_columns(
    raw_data["Hospital_General_Information.csv"]
)

hospital_scores = standardize_columns(
    raw_data["Timely_and_Effective_Care-Hospital.csv"]
)

state_scores = standardize_columns(
    raw_data["Timely_and_Effective_Care-State.csv"]
)

print("Working copies created.")
print(hospital_scores.columns.tolist())

Working copies created.
['facility_id', 'facility_name', 'address', 'city_town', 'state', 'zip_code', 'county_parish', 'telephone_number', 'condition', 'measure_id', 'measure_name', 'score', 'sample', 'footnote', 'start_date', 'end_date']


In [ ]:
for df in [hospital_general, hospital_scores, state_scores]:
    df["state"] = df["state"].str.strip().str.upper()

    if "facility_id" in df.columns:
        df["facility_id"] = df["facility_id"].str.strip()

    if "measure_id" in df.columns:
        df["measure_id"] = df["measure_id"].str.strip()

print("Extra spaces removed from state, facility, and measure keys.")

Extra spaces removed from state, facility, and measure keys.


In [ ]:
target_states = ["MS", "LA", "AL"]
target_measures = ["OP_18b", "OP_22"]

hospital_target = hospital_scores.loc[
    hospital_scores["state"].isin(target_states)
    & hospital_scores["measure_id"].isin(target_measures)
].copy()

state_target = state_scores.loc[
    state_scores["state"].isin(target_states)
    & state_scores["measure_id"].isin(target_measures)
].copy()

print("HOSPITAL MEASURES AND REPORTING PERIODS")

display(
    hospital_target[
        ["state", "measure_id", "measure_name", "start_date", "end_date"]
    ]
    .drop_duplicates()
    .sort_values(["measure_id", "state", "start_date"])
)

print("STATE BENCHMARKS AND REPORTING PERIODS")

display(
    state_target[
        ["state", "measure_id", "measure_name",
         "score", "start_date", "end_date"]
    ]
    .sort_values(["measure_id", "state", "start_date"])
)

HOSPITAL MEASURES AND REPORTING PERIODS


,state,measure_id,measure_name,start_date,end_date
11,AL,OP_18b,Average (median) time patients spent in the em...,10/01/2024,09/30/2025
51433,LA,OP_18b,Average (median) time patients spent in the em...,10/01/2024,09/30/2025
66371,MS,OP_18b,Average (median) time patients spent in the em...,10/01/2024,09/30/2025
14,AL,OP_22,Left before being seen,01/01/2024,12/31/2024
51436,LA,OP_22,Left before being seen,01/01/2024,12/31/2024
66374,MS,OP_22,Left before being seen,01/01/2024,12/31/2024


STATE BENCHMARKS AND REPORTING PERIODS


,state,measure_id,measure_name,score,start_date,end_date
37,AL,OP_18b,Average (median) time patients spent in the em...,149,10/01/2024,09/30/2025
626,LA,OP_18b,Average (median) time patients spent in the em...,129,10/01/2024,09/30/2025
874,MS,OP_18b,Average (median) time patients spent in the em...,129,10/01/2024,09/30/2025
52,AL,OP_22,Percentage of patients who left the emergency ...,2,01/01/2024,12/31/2024
641,LA,OP_22,Percentage of patients who left the emergency ...,2,01/01/2024,12/31/2024
889,MS,OP_22,Percentage of patients who left the emergency ...,3,01/01/2024,12/31/2024


In [ ]:
print("HOSPITAL SCORE VALUES — MOST FREQUENT FIRST")

display(
    hospital_target["score"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("original_score")
    .reset_index(name="record_count")
)

HOSPITAL SCORE VALUES — MOST FREQUENT FIRST


,original_score,record_count
0,Not Available,129
1,1,86
2,2,62
3,3,28
4,0,20
5,4,13
6,6,8
7,147,8
8,124,6
9,99,5


Data Cleaning and JOIN-Key Validation

Prepare working datasets for the three selected states.
Preserve original score values, convert scores to numbers, parse
reporting dates, and check whether JOIN keys are complete and unique.

In [ ]:
hospital_general_clean = hospital_general.loc[
    hospital_general["state"].isin(target_states)
].copy()

hospital_scores_clean = hospital_target.copy()
state_scores_clean = state_target.copy()

# Remove surrounding spaces from text values.
for df in [
    hospital_general_clean,
    hospital_scores_clean,
    state_scores_clean
]:
    for column in df.columns:
        df[column] = df[column].str.strip()

print("Working tables created for MS, LA, and AL.")

Working tables created for MS, LA, and AL.


In [ ]:
for df in [hospital_scores_clean, state_scores_clean]:

    # Preserve the original score for documentation.
    df["score_raw"] = df["score"]

    # Numeric scores become numbers.
    # Unavailable or nonnumeric values become missing.
    df["score"] = pd.to_numeric(
        df["score_raw"],
        errors="coerce"
    )

    # Preserve the original date text before conversion.
    for column in ["start_date", "end_date"]:
        df[f"{column}_raw"] = df[column]

        df[column] = pd.to_datetime(
            df[column],
            format="%m/%d/%Y",
            errors="coerce"
        )

print("Score and date conversions completed.")

Score and date conversions completed.


In [ ]:
for name, df in [
    ("Hospital scores", hospital_scores_clean),
    ("State scores", state_scores_clean)
]:
    print(f"\n{name}")
    print(f"Rows: {len(df):,}")
    print(f"Missing numeric scores: {df['score'].isna().sum()}")
    print(f"Missing or invalid start dates: {df['start_date'].isna().sum()}")
    print(f"Missing or invalid end dates: {df['end_date'].isna().sum()}")

    print("\nOriginal values converted to missing scores:")
    display(
        df.loc[df["score"].isna(), "score_raw"]
        .value_counts(dropna=False)
        .rename_axis("original_value")
        .reset_index(name="record_count")
    )

print("\nHOSPITAL SCORE COVERAGE BY STATE AND MEASURE")

display(
    hospital_scores_clean
    .groupby(["state", "measure_id"])
    .agg(
        total_records=("score", "size"),
        available_scores=("score", "count"),
        missing_scores=("score", lambda values: values.isna().sum())
    )
    .reset_index()
)


Hospital scores
Rows: 592
Missing numeric scores: 129
Missing or invalid start dates: 0
Missing or invalid end dates: 0

Original values converted to missing scores:


,original_value,record_count
0,Not Available,129



State scores
Rows: 6
Missing numeric scores: 0
Missing or invalid start dates: 0
Missing or invalid end dates: 0

Original values converted to missing scores:


,original_value,record_count



HOSPITAL SCORE COVERAGE BY STATE AND MEASURE


,state,measure_id,total_records,available_scores,missing_scores
0,AL,OP_18b,87,79,8
1,AL,OP_22,87,73,14
2,LA,OP_18b,112,89,23
3,LA,OP_22,112,88,24
4,MS,OP_18b,97,68,29
5,MS,OP_22,97,66,31


In [ ]:
checks = [
    (
        "Hospital general information",
        hospital_general_clean,
        ["facility_id"]
    ),
    (
        "Hospital scores",
        hospital_scores_clean,
        ["facility_id", "measure_id", "start_date", "end_date"]
    ),
    (
        "State benchmarks",
        state_scores_clean,
        ["state", "measure_id", "start_date", "end_date"]
    )
]

for name, df, keys in checks:
    missing_key = (
        df[keys].isna()
        | df[keys].eq("").fillna(False)
    ).any(axis=1)

    duplicate_key = df.duplicated(subset=keys, keep=False)

    print(f"\n{name}")
    print(f"Total rows: {len(df):,}")
    print(f"Rows with missing JOIN keys: {missing_key.sum()}")
    print(f"Rows sharing duplicate JOIN keys: {duplicate_key.sum()}")

    if duplicate_key.any():
        display(
            df.loc[duplicate_key]
            .sort_values(keys)
            .head(10)
        )


Hospital general information
Total rows: 369
Rows with missing JOIN keys: 0
Rows sharing duplicate JOIN keys: 0

Hospital scores
Total rows: 592
Rows with missing JOIN keys: 0
Rows sharing duplicate JOIN keys: 0

State benchmarks
Total rows: 6
Rows with missing JOIN keys: 0
Rows sharing duplicate JOIN keys: 0


Export Prepared Datasets

Export three separate tables for MySQL and Tableau.
Use standardized column names and YYYY-MM-DD dates.
Represent unavailable numeric scores as blank CSV fields and retain
score_raw for verification.

In [ ]:
export_tables = {
    "hospital_general_clean.csv": hospital_general_clean,
    "hospital_scores_clean.csv": hospital_scores_clean,
    "state_scores_clean.csv": state_scores_clean
}

for filename, df in export_tables.items():
    df.to_csv(
        filename,
        index=False,
        na_rep="",
        date_format="%Y-%m-%d",
        encoding="utf-8"
    )

    print(f"Exported {filename}: {len(df):,} rows")

Exported hospital_general_clean.csv: 369 rows
Exported hospital_scores_clean.csv: 592 rows
Exported state_scores_clean.csv: 6 rows


In [ ]:
import zipfile
from google.colab import files

zip_name = "healthcare_ed_prepared_data.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as archive:
    for filename in export_tables:
        archive.write(filename)

files.download(zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
for table_name, df in tables_for_bigquery.items():
    print(f"\nSCHEMA FOR: {table_name}")
    print(", ".join(f"{column}:STRING" for column in df.columns))


SCHEMA FOR: hospital_general_clean
facility_id:STRING, facility_name:STRING, address:STRING, city_town:STRING, state:STRING, zip_code:STRING, county_parish:STRING, telephone_number:STRING, hospital_type:STRING, hospital_ownership:STRING, emergency_services:STRING, meets_criteria_for_birthing_friendly_designation:STRING, hospital_overall_rating:STRING, hospital_overall_rating_footnote:STRING, mort_group_measure_count:STRING, count_of_facility_mort_measures:STRING, count_of_mort_measures_better:STRING, count_of_mort_measures_no_different:STRING, count_of_mort_measures_worse:STRING, mort_group_footnote:STRING, safety_group_measure_count:STRING, count_of_facility_safety_measures:STRING, count_of_safety_measures_better:STRING, count_of_safety_measures_no_different:STRING, count_of_safety_measures_worse:STRING, safety_group_footnote:STRING, readm_group_measure_count:STRING, count_of_facility_readm_measures:STRING, count_of_readm_measures_better:STRING, count_of_readm_measures_no_different:S